# 05_xray_console — State x-ray via OpenTelemetry (console output)

`OpenTelemetryPlugin` wires into `ActionMachine` as a plugin.
**No changes to Action code** — all observability is in the machine setup.

Two optional signals, both can be combined:

| Signal | Provider | What you get |
|--------|----------|-------------|
| **Traces** | `tracer_provider` | Root span + child span per aspect; timings, status |
| **Logs (x-ray)** | `logger_provider` | Log record per lifecycle event; `aoa.state.*` attributes after each step |

This notebook uses `ConsoleSpanExporter` and `ConsoleLogRecordExporter`
so you can see everything in the cell output without any backend.

In [ ]:
!pip install "aoa-action-machine[otel]"

In [ ]:
import asyncio

from opentelemetry.sdk._logs import LoggerProvider
from opentelemetry.sdk._logs.export import ConsoleLogRecordExporter, SimpleLogRecordProcessor
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor
from pydantic import Field

from aoa.action_machine.auth import GuestRole
from aoa.action_machine.context import Context
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.aspects import regular_aspect, summary_aspect
from aoa.action_machine.intents.check_roles import check_roles
from aoa.action_machine.intents.checkers import result_float, result_int, result_string
from aoa.action_machine.intents.meta import meta
from aoa.action_machine.model import BaseAction, BaseParams, BaseResult
from aoa.action_machine.plugin.open_telemetry import OpenTelemetryPlugin
from aoa.action_machine.runtime.action_product_machine import ActionProductMachine

## Domain, Params, Result

In [ ]:
class CheckoutDomain(BaseDomain):
    name = "checkout"
    description = "E-commerce checkout pipeline"


class CheckoutParams(BaseParams):
    sku: str = Field(description="Product SKU")
    quantity: int = Field(description="Number of units")
    unit_price: float = Field(description="Price per unit in USD")


class CheckoutResult(BaseResult):
    sku: str = Field(description="Validated SKU")
    total: float = Field(description="Order total in USD")
    confirmation: str = Field(description="Human-readable confirmation message")

## Action — three steps, each with checked state

After `validate_aspect`: `state = {"sku": ..., "quantity": ...}`
After `enrich_aspect`: `state = {"sku": ..., "quantity": ..., "total": ...}`

The x-ray log records will show exactly these fields at each step.

In [ ]:
@meta(description="Three-step checkout: validate → enrich → confirm", domain=CheckoutDomain)
@check_roles(GuestRole)
class CheckoutAction(BaseAction[CheckoutParams, CheckoutResult]):

    @regular_aspect("Step 1: Validate SKU and quantity")
    @result_string("sku", required=True, min_length=3)
    @result_int("quantity", required=True, min_value=1, max_value=500)
    async def validate_aspect(self, params, state, box, connections):
        return {"sku": params.sku.strip().upper(), "quantity": params.quantity}

    @regular_aspect("Step 2: Calculate total")
    @result_string("sku", required=True)
    @result_int("quantity", required=True)
    @result_float("total", required=True, min_value=0.01)
    async def enrich_aspect(self, params, state, box, connections):
        return {
            "sku": state["sku"],
            "quantity": state["quantity"],
            "total": round(state["quantity"] * params.unit_price, 2),
        }

    @summary_aspect("Step 3: Confirm order")
    async def confirm_summary(self, params, state, box, connections):
        return CheckoutResult(
            sku=state["sku"],
            total=state["total"],
            confirmation=f"Order confirmed: {state['quantity']}x {state['sku']} = ${state['total']}",
        )

## OTel setup — console exporters

Both providers are optional. Using both together enables automatic log-to-trace
correlation: every log record carries the same `trace_id` as the root span,
and after-step logs carry the `span_id` of the corresponding child span.

In [ ]:
tp = TracerProvider()
tp.add_span_processor(SimpleSpanProcessor(ConsoleSpanExporter()))

lp = LoggerProvider()
lp.add_log_record_processor(SimpleLogRecordProcessor(ConsoleLogRecordExporter()))

plugin = OpenTelemetryPlugin(
    tracer_provider=tp,
    logger_provider=lp,
    service_name="checkout-service",
)

## Run

Watch the output for two types of JSON objects:
- Spans: `"name": "validate_aspect"` — timing and structure
- Logs: `"body": "aoa.aspect.regular.after"` — state x-ray with `aoa.state.*` attributes

After `validate_aspect` you'll see `aoa.state.sku` and `aoa.state.quantity`.
After `enrich_aspect` you'll see those two plus `aoa.state.total`.
That's the x-ray: each record shows the exact state after that step.

In [ ]:
machine = ActionProductMachine(plugins=[plugin])
result = await machine.run(
    Context(),
    CheckoutAction(),
    CheckoutParams(sku=" widget-42 ", quantity=3, unit_price=19.99),
)
print(f"\nFinal result: {result.confirmation}")